# Olist transform notebook (raw → curated)
Reads the raw files landed by ADF in ADLS (`raw/<entity>/run_id=<id>/` —
parquet for the five transactional extracts, csv snapshots for the four
reference files), applies the transformations defined in
`olist_transforms` (rename / cast / null handling / dedup / DQ flags /
ingestion metadata), writes typed tables to Azure SQL via JDBC, and logs
per-entity row counts to `etl.pipeline_process_log`.

Parameters are passed by the ADF Notebook activity via widgets.
Import BOTH files into the same workspace folder (e.g. `/Shared/`):
this notebook `%run`s `./olist_transforms`.

In [ ]:
from datetime import datetime, timezone

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [ ]:
# Parameters (supplied by ADF: @pipeline().RunId and the raw folder path).
# `only_entity` is a debug aid for the interactive smoke test: set it to a
# single entity name (e.g. product_category_translation) to process just
# that one; leave empty for the full 9-entity run.

dbutils.widgets.text("run_id", "manual-local-run")
dbutils.widgets.text("raw_base_path", "abfss://raw@<storageaccount>.dfs.core.windows.net")
dbutils.widgets.text("only_entity", "")

RUN_ID = dbutils.widgets.get("run_id")
RAW = dbutils.widgets.get("raw_base_path").rstrip("/")
ONLY_ENTITY = dbutils.widgets.get("only_entity").strip()

In [ ]:
# ADLS access — storage account key from the Key-Vault-backed secret scope.
# (Exercise-grade auth; Unity Catalog external locations / a service
# principal are the production answer — see README.)
# The account name is parsed from raw_base_path: abfss://raw@<acct>.dfs...

STORAGE_ACCOUNT = RAW.split("@")[1].split(".")[0]
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get("kv-olist", "storage-key"),
)

In [ ]:
# JDBC target config — secrets from the same scope

JDBC_URL = (
    "jdbc:sqlserver://{server}.database.windows.net:1433;"
    "database={db};encrypt=true;trustServerCertificate=false;loginTimeout=30;"
).format(
    server=dbutils.secrets.get("kv-olist", "sql-server-name"),
    db=dbutils.secrets.get("kv-olist", "sql-db-name"),
)
JDBC_PROPS = {
    "user": dbutils.secrets.get("kv-olist", "sql-user"),
    "password": dbutils.secrets.get("kv-olist", "sql-password"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}


def write_curated(df: DataFrame, table: str) -> None:
    """Truncate-load into a pre-created Azure SQL table.

    mode('overwrite') + truncate=true keeps the DDL (types, constraints)
    defined in 01_azure_sql_ddl.sql instead of letting Spark drop and
    recreate the table with inferred types.
    """
    (
        df.write.format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", table)
        .option("user", JDBC_PROPS["user"])
        .option("password", JDBC_PROPS["password"])
        .option("driver", JDBC_PROPS["driver"])
        .option("truncate", "true")
        .option("batchsize", 10000)
        .mode("overwrite")
        .save()
    )


def log_process(entity: str, stage: str, rows_read, rows_written,
                rows_flagged, status: str, message: str,
                started_at: datetime, finished_at: datetime) -> None:
    """Append one row to etl.pipeline_process_log."""
    schema = T.StructType([
        T.StructField("pipeline_run_id", T.StringType()),
        T.StructField("entity_name", T.StringType()),
        T.StructField("stage", T.StringType()),
        T.StructField("rows_read", T.LongType()),
        T.StructField("rows_written", T.LongType()),
        T.StructField("rows_flagged", T.LongType()),
        T.StructField("status", T.StringType()),
        T.StructField("message", T.StringType()),
        T.StructField("started_at_utc", T.TimestampType()),
        T.StructField("finished_at_utc", T.TimestampType()),
    ])
    row = [(RUN_ID, entity, stage, rows_read, rows_written, rows_flagged,
            status, message, started_at, finished_at)]
    (
        spark.createDataFrame(row, schema)
        .write.format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", "etl.pipeline_process_log")
        .option("user", JDBC_PROPS["user"])
        .option("password", JDBC_PROPS["password"])
        .option("driver", JDBC_PROPS["driver"])
        .mode("append")
        .save()
    )

In [ ]:
%run ./olist_transforms

## The 8 uniform entities (config-driven)
One loop over `ENTITY_CONFIG`. Per entity: read raw → transform →
JDBC truncate-load → process log. A failing entity is logged as
FAILED and the loop continues (failure isolation); the run is
failed at the end so ADF Monitor still shows red.

In [ ]:
def read_raw(entity: str, cfg: dict) -> DataFrame:
    path = f"{RAW}/{entity}/run_id={RUN_ID}/"
    if cfg["source_kind"] == "csv":
        return spark.read.schema(cfg["csv_schema"]).option("header", "true").csv(path)
    return spark.read.parquet(path)


failures = []
n_processed = 0

for entity, cfg in ENTITY_CONFIG.items():
    if ONLY_ENTITY and entity != ONLY_ENTITY:
        continue
    t0 = datetime.now(timezone.utc)
    try:
        df_raw = read_raw(entity, cfg)
        n_read = df_raw.count()
        df_out = apply_transforms(df_raw, cfg, RUN_ID)
        n_written = df_out.count()
        n_flagged = count_flagged(df_out, cfg)
        write_curated(df_out, cfg["target"])
        log_process(entity, "curated_write", n_read, n_written, n_flagged,
                    "SUCCESS", f"{cfg['source_kind']} → {cfg['target']}",
                    t0, datetime.now(timezone.utc))
        n_processed += 1
        print(f"[OK]     {entity:<30} read={n_read:>9,}  written={n_written:>9,}  flagged={n_flagged}")
    except Exception as exc:
        failures.append(entity)
        log_process(entity, "curated_write", None, None, None,
                    "FAILED", str(exc)[:900], t0, datetime.now(timezone.utc))
        print(f"[FAILED] {entity}: {exc}")

## Geolocation (bespoke: grain-changing aggregation)
1,000,163 raw points → one row per zip prefix (~19k) via
`aggregate_geolocation`. `rows_flagged` = points excluded from the
centroids by the Brazil bounding box (counted per zip in
`invalid_point_count`). Full point-level fidelity stays in raw.

In [ ]:
if not ONLY_ENTITY or ONLY_ENTITY == "geolocation":
    t0 = datetime.now(timezone.utc)
    try:
        geo_raw = (
            spark.read.schema(GEOLOCATION_SCHEMA)
            .option("header", "true")
            .csv(f"{RAW}/geolocation/run_id={RUN_ID}/")
        )
        n_read = geo_raw.count()
        geo_dim = aggregate_geolocation(geo_raw, RUN_ID)
        n_written = geo_dim.count()
        n_flagged = geo_dim.agg(F.sum("invalid_point_count")).collect()[0][0] or 0
        write_curated(geo_dim, "curated.geolocation")
        log_process("geolocation", "curated_write", n_read, n_written, n_flagged,
                    "SUCCESS", "csv → curated.geolocation (zip-grain dim)",
                    t0, datetime.now(timezone.utc))
        n_processed += 1
        print(f"[OK]     {'geolocation':<30} read={n_read:>9,}  written={n_written:>9,}  flagged={n_flagged}")
    except Exception as exc:
        failures.append("geolocation")
        log_process("geolocation", "curated_write", None, None, None,
                    "FAILED", str(exc)[:900], t0, datetime.now(timezone.utc))
        print(f"[FAILED] geolocation: {exc}")

## Run outcome

In [ ]:
if failures:
    raise Exception(f"run_id={RUN_ID}: {len(failures)} entities FAILED: {', '.join(failures)} "
                    f"(successful entities are loaded and logged; see etl.pipeline_process_log)")

dbutils.notebook.exit(f"SUCCESS run_id={RUN_ID} entities={n_processed}")